# Predicting Renewable Energy Output with the Prophet Model

Prophet, by Meta, is a procedure model for forecasting time series data based on an additive model where non-linear trends are fit with yearly, weekly, and daily seasonality, plus holiday effects. Given the hourly nature of our data with clear temporal structure, Prophet is good choice because of design specific forecasting strengths for time series data. 

In [67]:
from prophet import Prophet
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [68]:
df = pd.read_csv('../data/processed/Feature_Engineering _Dataset/feature_engineered_data.csv')

## Solar Energy

In [69]:
df['Solar_lag_1'] = df['Solar_lag_1'].fillna(0)
df['Solar_lag_2'] = df['Solar_lag_2'].fillna(0)
df['Solar_lag_3'] = df['Solar_lag_3'].fillna(0)
df['Solar_lag_24'] = df['Solar_lag_24'].fillna(0)
df['Solar_rolling_24h_mean'] = df['Solar_rolling_24h_mean'].fillna(0)
df['Solar_rolling_24h_std'] = df['Solar_rolling_24h_std'].fillna(0)

For the training dataset, the first 24 rows were excluded. Thes first 24 rows correspond to the first day of the recorded data so certain features representing lagged values are NaN and do not contain values until past a certain hour. The first 'Solar_lag_24' value appears after the first 24 hours hence the reason for exluding those rows.

For the testing dataset, the last 168 rows only. This corresponds exactly to a period of one (1) week and this portion will be used to test the model.

In [70]:
df_solar_train = df.iloc[24:]
df_solar_test = df.iloc[-168:]

Now that the training and testing dataframes are partioned, in order to prepare the training dataframe for the model the relevant features that are geared toward Solar Energy output are selected here. 'Time' column renamed 'ds' and 'Solar' column renamed 'y' to represent the target.

In [71]:
df_solar_train = df_solar_train[['time', 'Solar', 'temp', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'Solar_lag_1', 'Solar_lag_2', 'Solar_lag_3', 'Solar_lag_24', 'Solar_rolling_24h_mean', 'Solar_rolling_24h_std']].rename(columns={'time': 'ds', 'Solar': 'y'})
df_solar_train.head(10)

,ds,y,temp,hour_sin,hour_cos,month_sin,month_cos,Solar_lag_1,Solar_lag_2,Solar_lag_3,Solar_lag_24,Solar_rolling_24h_mean,Solar_rolling_24h_std
24,2016-01-02 00:00:00,0.0,7.4,0.000000,1.000000e+00,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
25,2016-01-02 01:00:00,0.0,7.3,0.258819,9.659258e-01,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
26,2016-01-02 02:00:00,0.0,7.9,0.500000,8.660254e-01,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
27,2016-01-02 03:00:00,0.0,8.1,0.707107,7.071068e-01,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
28,2016-01-02 04:00:00,0.0,8.4,0.866025,5.000000e-01,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
29,2016-01-02 05:00:00,0.0,8.6,0.965926,2.588190e-01,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
30,2016-01-02 06:00:00,0.0,8.6,1.000000,6.123234e-17,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
31,2016-01-02 07:00:00,0.0,8.7,0.965926,-2.588190e-01,0.5,0.866025,0.0,0.0,0.0,0.0,317.833333,515.471558
32,2016-01-02 08:00:00,87.0,9.0,0.866025,-5.000000e-01,0.5,0.866025,0.0,0.0,0.0,100.0,317.291667,515.717186
33,2016-01-02 09:00:00,274.0,9.5,0.707107,-7.071068e-01,0.5,0.866025,87.0,0.0,0.0,356.0,313.875000,515.721218


The features added to the model to make predictions:
- temp: hourly temperature which is a high indicator of solar output
- hour_sin, hour_cos to capture the daily cyclical patterns in a gradual manner
- month_sin, month_cos to capture yearly seasonality such as longer days in summer which may point to more solar energy
- solar_lag_1/2/3/24 for past values of solar energy output which helps to model short-term patterns
- solar_rolling_24h_mean/std for trend and variability in the past day wich helps with modeling recent stability or fluctuation

In [72]:
model = Prophet()

model.add_regressor('temp')
model.add_regressor('hour_sin')
model.add_regressor('hour_cos')
model.add_regressor('month_sin')
model.add_regressor('month_cos')
model.add_regressor('Solar_lag_1')
model.add_regressor('Solar_lag_2')
model.add_regressor('Solar_lag_3')
model.add_regressor('Solar_lag_24')
model.add_regressor('Solar_rolling_24h_mean')
model.add_regressor('Solar_rolling_24h_std')

In [73]:
model.fit(df_solar_train)

14:25:55 - cmdstanpy - INFO - Chain [1] start processing
14:26:17 - cmdstanpy - INFO - Chain [1] done processing


To prepare the testing dataframe for the model the relevant features that are geared toward Solar Energy output are selected here. Since 'Solar' or 'y' needs to be derived, a separate dataframe excluding that column will used by the model. The actual values of 'y' from the testing dataframe will be compared against the predicted values of 'y' from the dataset used by the model.

In [74]:
df_solar_test = df_solar_test[['time', 'Solar', 'temp', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'Solar_lag_1', 'Solar_lag_2', 'Solar_lag_3', 'Solar_lag_24', 'Solar_rolling_24h_mean', 'Solar_rolling_24h_std']].rename(columns={'time': 'ds', 'Solar': 'y'})
df_solar_test_features = df_solar_test.drop(columns=['y'])

In [75]:
df_solar_test.head(10)

,ds,y,temp,hour_sin,hour_cos,month_sin,month_cos,Solar_lag_1,Solar_lag_2,Solar_lag_3,Solar_lag_24,Solar_rolling_24h_mean,Solar_rolling_24h_std
177042,2022-09-20 00:00:00,48.0,19.5,0.000000,1.000000e+00,-1.0,-1.836970e-16,104.0,140.0,244.0,44.0,3944.000000,4794.883360
177043,2022-09-20 01:00:00,48.0,19.7,0.258819,9.659258e-01,-1.0,-1.836970e-16,48.0,104.0,140.0,28.0,3944.833333,4794.174868
177044,2022-09-20 02:00:00,48.0,20.4,0.500000,8.660254e-01,-1.0,-1.836970e-16,48.0,48.0,104.0,28.0,3945.666667,4793.466121
177045,2022-09-20 03:00:00,48.0,19.4,0.707107,7.071068e-01,-1.0,-1.836970e-16,48.0,48.0,48.0,28.0,3946.500000,4792.757117
177046,2022-09-20 04:00:00,40.0,19.5,0.866025,5.000000e-01,-1.0,-1.836970e-16,48.0,48.0,48.0,24.0,3947.166667,4792.188859
177047,2022-09-20 05:00:00,36.0,20.6,0.965926,2.588190e-01,-1.0,-1.836970e-16,40.0,48.0,48.0,12.0,3948.166667,4791.334421
177048,2022-09-20 06:00:00,40.0,21.2,1.000000,6.123234e-17,-1.0,-1.836970e-16,36.0,40.0,48.0,12.0,3949.333333,4790.337618
177049,2022-09-20 07:00:00,36.0,25.0,0.965926,-2.588190e-01,-1.0,-1.836970e-16,40.0,36.0,40.0,12.0,3950.333333,4789.482378
177050,2022-09-20 08:00:00,40.0,27.3,0.866025,-5.000000e-01,-1.0,-1.836970e-16,36.0,40.0,36.0,12.0,3951.500000,4788.484638
177051,2022-09-20 09:00:00,2008.0,27.2,0.707107,-7.071068e-01,-1.0,-1.836970e-16,40.0,36.0,40.0,1236.0,3983.666667,4772.014784


In [76]:
df_solar_test_features.head(10)

,ds,temp,hour_sin,hour_cos,month_sin,month_cos,Solar_lag_1,Solar_lag_2,Solar_lag_3,Solar_lag_24,Solar_rolling_24h_mean,Solar_rolling_24h_std
177042,2022-09-20 00:00:00,19.5,0.000000,1.000000e+00,-1.0,-1.836970e-16,104.0,140.0,244.0,44.0,3944.000000,4794.883360
177043,2022-09-20 01:00:00,19.7,0.258819,9.659258e-01,-1.0,-1.836970e-16,48.0,104.0,140.0,28.0,3944.833333,4794.174868
177044,2022-09-20 02:00:00,20.4,0.500000,8.660254e-01,-1.0,-1.836970e-16,48.0,48.0,104.0,28.0,3945.666667,4793.466121
177045,2022-09-20 03:00:00,19.4,0.707107,7.071068e-01,-1.0,-1.836970e-16,48.0,48.0,48.0,28.0,3946.500000,4792.757117
177046,2022-09-20 04:00:00,19.5,0.866025,5.000000e-01,-1.0,-1.836970e-16,48.0,48.0,48.0,24.0,3947.166667,4792.188859
177047,2022-09-20 05:00:00,20.6,0.965926,2.588190e-01,-1.0,-1.836970e-16,40.0,48.0,48.0,12.0,3948.166667,4791.334421
177048,2022-09-20 06:00:00,21.2,1.000000,6.123234e-17,-1.0,-1.836970e-16,36.0,40.0,48.0,12.0,3949.333333,4790.337618
177049,2022-09-20 07:00:00,25.0,0.965926,-2.588190e-01,-1.0,-1.836970e-16,40.0,36.0,40.0,12.0,3950.333333,4789.482378
177050,2022-09-20 08:00:00,27.3,0.866025,-5.000000e-01,-1.0,-1.836970e-16,36.0,40.0,36.0,12.0,3951.500000,4788.484638
177051,2022-09-20 09:00:00,27.2,0.707107,-7.071068e-01,-1.0,-1.836970e-16,40.0,36.0,40.0,1236.0,3983.666667,4772.014784


With the model trained on the regressors, it is now used to generate forecasts on the test features

In [77]:
forecast = model.predict(df_solar_test_features)
forecast.head(10)

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,Solar_lag_1,Solar_lag_1_lower,Solar_lag_1_upper,Solar_lag_2,...,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2022-09-20 00:00:00,1905.778783,-218.922916,575.321070,1905.778783,1905.778783,-2877.127438,-2877.127438,-2877.127438,1844.878205,...,1.669107,1.669107,1.669107,-7.757290,-7.757290,-7.757290,0.0,0.0,0.0,178.513703
1,2022-09-20 01:00:00,1905.779102,-237.565187,525.180216,1905.779102,1905.779102,-2966.859816,-2966.859816,-2966.859816,1882.624012,...,1.812016,1.812016,1.812016,-7.749690,-7.749690,-7.749690,0.0,0.0,0.0,148.453853
2,2022-09-20 02:00:00,1905.779421,-156.449706,587.076073,1905.779421,1905.779421,-2966.859816,-2966.859816,-2966.859816,1941.339711,...,1.947466,1.947466,1.947466,-7.742115,-7.742115,-7.742115,0.0,0.0,0.0,222.586797
3,2022-09-20 03:00:00,1905.779741,-210.296050,530.065290,1905.779741,1905.779741,-2966.859816,-2966.859816,-2966.859816,1941.339711,...,2.075695,2.075695,2.075695,-7.734567,-7.734567,-7.734567,0.0,0.0,0.0,169.757916
4,2022-09-20 04:00:00,1905.780060,-279.742459,478.515466,1905.780060,1905.780060,-2966.859816,-2966.859816,-2966.859816,1941.339711,...,2.196934,2.196934,2.196934,-7.727045,-7.727045,-7.727045,0.0,0.0,0.0,110.658081
5,2022-09-20 05:00:00,1905.780379,-264.412219,470.925526,1905.780379,1905.780379,-2979.678728,-2979.678728,-2979.678728,1941.339711,...,2.311400,2.311400,2.311400,-7.719550,-7.719550,-7.719550,0.0,0.0,0.0,111.611769
6,2022-09-20 06:00:00,1905.780698,-135.681681,630.822680,1905.780698,1905.780698,-2986.088183,-2986.088183,-2986.088183,1949.727668,...,2.419279,2.419279,2.419279,-7.712081,-7.712081,-7.712081,0.0,0.0,0.0,256.894214
7,2022-09-20 07:00:00,1905.781017,116.789514,865.168316,1905.781017,1905.781017,-2979.678728,-2979.678728,-2979.678728,1953.921647,...,2.520718,2.520718,2.520718,-7.704640,-7.704640,-7.704640,0.0,0.0,0.0,491.884249
8,2022-09-20 08:00:00,1905.781336,275.592323,1054.956435,1905.781336,1905.781336,-2986.088183,-2986.088183,-2986.088183,1949.727668,...,2.615815,2.615815,2.615815,-7.697227,-7.697227,-7.697227,0.0,0.0,0.0,666.424110
9,2022-09-20 09:00:00,1905.781655,514.988247,1267.302520,1905.781655,1905.781655,-2979.678728,-2979.678728,-2979.678728,1953.921647,...,2.704606,2.704606,2.704606,-7.689841,-7.689841,-7.689841,0.0,0.0,0.0,904.520890


### Metrics

An evaluation of its performance by assessing how well the model predicted solar energy output by comparing the actual values (y_true) with the predicted values (y_pred). The three (3) metrics for this evaluation are: Mean Absolute Error, Root Mean Squared Error and R-Squared (R²)

In [78]:
y_true = df_solar_test['y'].values
y_pred = forecast['yhat'].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

Mean Absolute Error (MAE)

In [79]:
mae = round(mae, 2)
mae

553.99

Root Mean Squared Error (RMSE)

In [80]:
rmse = float(round(rmse, 2))
rmse

803.21

R-Squared (R²)

In [81]:
r2 = round(r2, 4)
r2

0.9717

## Wind Energy

In [82]:
df['Wind Onshore_lag_1'] = df['Wind Onshore_lag_1'].fillna(0)
df['Wind Onshore_lag_2'] = df['Wind Onshore_lag_2'].fillna(0)
df['Wind Onshore_lag_3'] = df['Wind Onshore_lag_3'].fillna(0)
df['Wind Onshore_lag_24'] = df['Wind Onshore_lag_24'].fillna(0)
df['Wind Onshore_rolling_24h_mean'] = df['Wind Onshore_rolling_24h_mean'].fillna(0)
df['Wind Onshore_rolling_24h_std'] = df['Wind Onshore_rolling_24h_std'].fillna(0)

For the training dataset, the first 24 rows were excluded. Thes first 24 rows correspond to the first day of the recorded data so certain features representing lagged values are NaN and do not contain values until past a certain hour. The first 'Solar_lag_24' value appears after the first 24 hours hence the reason for exluding those rows.

For the testing dataset, the last 168 rows only. This corresponds exactly to a period of one (1) week and this portion will be used to test the model.

In [83]:
df_wind_train = df.iloc[24:]
df_wind_test = df.iloc[-168:]

Now that the training and testing dataframes are partioned, in order to prepare the training dataframe for the model the relevant features that are geared toward Solar Energy output are selected here. 'Time' column renamed 'ds' and 'Wind Onshore' column renamed 'y' to represent the target.

In [84]:
df_wind_train = df_wind_train[['time', 'Wind Onshore', 'wspd', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'wind_power_potential', 'Wind Onshore_lag_1', 'Wind Onshore_lag_2', 'Wind Onshore_lag_3', 'Wind Onshore_lag_24', 'Wind Onshore_rolling_24h_mean', 'Wind Onshore_rolling_24h_std']].rename(columns={'time': 'ds', 'Wind Onshore': 'y'})
df_wind_train

,ds,y,wspd,hour_sin,hour_cos,month_sin,month_cos,wind_power_potential,Wind Onshore_lag_1,Wind Onshore_lag_2,Wind Onshore_lag_3,Wind Onshore_lag_24,Wind Onshore_rolling_24h_mean,Wind Onshore_rolling_24h_std
24,2016-01-02 00:00:00,5038.0,11.2,0.000000,1.000000,0.5,8.660254e-01,1404.928,5255.0,5353.0,5493.0,2189.0,3015.000000,1765.164508
25,2016-01-02 01:00:00,4724.0,11.2,0.258819,0.965926,0.5,8.660254e-01,1404.928,5038.0,5255.0,5353.0,1753.0,3138.791667,1776.951272
26,2016-01-02 02:00:00,4669.0,13.0,0.500000,0.866025,0.5,8.660254e-01,2197.000,4724.0,5038.0,5255.0,1430.0,3273.750000,1764.483204
27,2016-01-02 03:00:00,4612.0,13.0,0.707107,0.707107,0.5,8.660254e-01,2197.000,4669.0,4724.0,5038.0,1214.0,3415.333333,1727.972742
28,2016-01-02 04:00:00,4625.0,14.8,0.866025,0.500000,0.5,8.660254e-01,3241.792,4612.0,4669.0,4724.0,1031.0,3565.083333,1667.013886
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
177205,2022-09-26 19:00:00,7176.0,7.6,-0.965926,0.258819,-1.0,-1.836970e-16,438.976,6792.0,6212.0,5276.0,9024.0,7134.833333,2096.631992
177206,2022-09-26 20:00:00,6784.0,5.4,-0.866025,0.500000,-1.0,-1.836970e-16,157.464,7176.0,6792.0,6212.0,9856.0,7006.833333,2015.483881
177207,2022-09-26 21:00:00,7032.0,0.0,-0.707107,0.707107,-1.0,-1.836970e-16,0.000,6784.0,7176.0,6792.0,10096.0,6879.166667,1905.330547
177208,2022-09-26 22:00:00,7012.0,0.0,-0.500000,0.866025,-1.0,-1.836970e-16,0.000,7032.0,6784.0,7176.0,10020.0,6753.833333,1784.868176


The features added to the model to make predictions:
- wspd: hourly wind speed which is a high indicator of wind energy output
- hour_sin, hour_cos to capture the daily wind shifting patterns
- month_sin, month_cos to capture yearly seasonality as wind patterns vary with season
- wind_power_potential for an estimation of how efficiently wind energy can be converted to power
- wind onshore_lag_1/2/3/24 for past values of wind energy output which helps to model short-term patterns
- wind onshore_rolling_24h_mean/std for trend and variability in the past day wich helps with modeling recent stability or fluctuation

In [85]:
model = Prophet()

model.add_regressor('wspd')
model.add_regressor('hour_sin')
model.add_regressor('hour_cos')
model.add_regressor('month_sin')
model.add_regressor('month_cos')
model.add_regressor('wind_power_potential')
model.add_regressor('Wind Onshore_lag_1')
model.add_regressor('Wind Onshore_lag_2')
model.add_regressor('Wind Onshore_lag_3')
model.add_regressor('Wind Onshore_lag_24')
model.add_regressor('Wind Onshore_rolling_24h_mean')
model.add_regressor('Wind Onshore_rolling_24h_std')

In [86]:
model.fit(df_wind_train)

14:26:42 - cmdstanpy - INFO - Chain [1] start processing
14:28:06 - cmdstanpy - INFO - Chain [1] done processing


To prepare the testing dataframe for the model the relevant features that are geared toward Solar Energy output are selected here. Since 'Wind Onshore' or 'y' needs to be derived, a separate dataframe excluding that column will used by the model. The actual values of 'y' from the testing dataframe will be compared against the predicted values of 'y' from the dataset used by the model.

In [87]:
df_wind_test = df_wind_test[['time', 'Wind Onshore', 'wspd', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos', 'wind_power_potential', 'Wind Onshore_lag_1', 'Wind Onshore_lag_2', 'Wind Onshore_lag_3', 'Wind Onshore_lag_24', 'Wind Onshore_rolling_24h_mean', 'Wind Onshore_rolling_24h_std']].rename(columns={'time': 'ds', 'Wind Onshore': 'y'})
df_wind_test_features = df_wind_test.drop(columns=['y'])

In [88]:
df_wind_test.head(10)

,ds,y,wspd,hour_sin,hour_cos,month_sin,month_cos,wind_power_potential,Wind Onshore_lag_1,Wind Onshore_lag_2,Wind Onshore_lag_3,Wind Onshore_lag_24,Wind Onshore_rolling_24h_mean,Wind Onshore_rolling_24h_std
177042,2022-09-20 00:00:00,4888.0,11.2,0.000000,1.000000e+00,-1.0,-1.836970e-16,1404.928,4584.0,4392.0,4544.0,5280.0,3365.333333,1248.401088
177043,2022-09-20 01:00:00,5248.0,9.4,0.258819,9.659258e-01,-1.0,-1.836970e-16,830.584,4888.0,4584.0,4392.0,5136.0,3370.000000,1255.496991
177044,2022-09-20 02:00:00,5292.0,13.0,0.500000,8.660254e-01,-1.0,-1.836970e-16,2197.000,5248.0,4888.0,4584.0,5064.0,3379.500000,1269.655106
177045,2022-09-20 03:00:00,5220.0,9.4,0.707107,7.071068e-01,-1.0,-1.836970e-16,830.584,5292.0,5248.0,4888.0,4776.0,3398.000000,1293.891468
177046,2022-09-20 04:00:00,5252.0,11.2,0.866025,5.000000e-01,-1.0,-1.836970e-16,1404.928,5220.0,5292.0,5248.0,4360.0,3435.166667,1334.887837
177047,2022-09-20 05:00:00,4736.0,11.2,0.965926,2.588190e-01,-1.0,-1.836970e-16,1404.928,5252.0,5220.0,5292.0,4052.0,3463.666667,1355.768310
177048,2022-09-20 06:00:00,4428.0,11.2,1.000000,6.123234e-17,-1.0,-1.836970e-16,1404.928,4736.0,5252.0,5220.0,3808.0,3489.500000,1368.462069
177049,2022-09-20 07:00:00,4120.0,9.4,0.965926,-2.588190e-01,-1.0,-1.836970e-16,830.584,4428.0,4736.0,5252.0,3428.0,3518.333333,1374.387284
177050,2022-09-20 08:00:00,3844.0,24.1,0.866025,-5.000000e-01,-1.0,-1.836970e-16,13997.521,4120.0,4428.0,4736.0,3248.0,3543.166667,1374.674781
177051,2022-09-20 09:00:00,3656.0,25.9,0.707107,-7.071068e-01,-1.0,-1.836970e-16,17373.979,3844.0,4120.0,4428.0,2804.0,3578.666667,1365.728430


In [89]:
df_wind_test_features

,ds,wspd,hour_sin,hour_cos,month_sin,month_cos,wind_power_potential,Wind Onshore_lag_1,Wind Onshore_lag_2,Wind Onshore_lag_3,Wind Onshore_lag_24,Wind Onshore_rolling_24h_mean,Wind Onshore_rolling_24h_std
177042,2022-09-20 00:00:00,11.2,0.000000,1.000000,-1.0,-1.836970e-16,1404.928,4584.0,4392.0,4544.0,5280.0,3365.333333,1248.401088
177043,2022-09-20 01:00:00,9.4,0.258819,0.965926,-1.0,-1.836970e-16,830.584,4888.0,4584.0,4392.0,5136.0,3370.000000,1255.496991
177044,2022-09-20 02:00:00,13.0,0.500000,0.866025,-1.0,-1.836970e-16,2197.000,5248.0,4888.0,4584.0,5064.0,3379.500000,1269.655106
177045,2022-09-20 03:00:00,9.4,0.707107,0.707107,-1.0,-1.836970e-16,830.584,5292.0,5248.0,4888.0,4776.0,3398.000000,1293.891468
177046,2022-09-20 04:00:00,11.2,0.866025,0.500000,-1.0,-1.836970e-16,1404.928,5220.0,5292.0,5248.0,4360.0,3435.166667,1334.887837
...,...,...,...,...,...,...,...,...,...,...,...,...,...
177205,2022-09-26 19:00:00,7.6,-0.965926,0.258819,-1.0,-1.836970e-16,438.976,6792.0,6212.0,5276.0,9024.0,7134.833333,2096.631992
177206,2022-09-26 20:00:00,5.4,-0.866025,0.500000,-1.0,-1.836970e-16,157.464,7176.0,6792.0,6212.0,9856.0,7006.833333,2015.483881
177207,2022-09-26 21:00:00,0.0,-0.707107,0.707107,-1.0,-1.836970e-16,0.000,6784.0,7176.0,6792.0,10096.0,6879.166667,1905.330547
177208,2022-09-26 22:00:00,0.0,-0.500000,0.866025,-1.0,-1.836970e-16,0.000,7032.0,6784.0,7176.0,10020.0,6753.833333,1784.868176


With the model trained on the regressors, it is now used to generate forecasts on the test features

In [90]:
forecast = model.predict(df_wind_test_features)
forecast

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,Wind Onshore_lag_1,Wind Onshore_lag_1_lower,Wind Onshore_lag_1_upper,Wind Onshore_lag_2,...,wspd,wspd_lower,wspd_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2022-09-20 00:00:00,3812.128130,4342.812838,4949.029353,3812.128130,3812.128130,1229.753760,1229.753760,1229.753760,-341.532905,...,-2.472116,-2.472116,-2.472116,1.975775,1.975775,1.975775,0.0,0.0,0.0,4645.573016
1,2022-09-20 01:00:00,3812.128263,4676.718700,5321.200608,3812.128263,3812.128263,1711.030991,1711.030991,1711.030991,-453.655681,...,-8.261207,-8.261207,-8.261207,1.991686,1.991686,1.991686,0.0,0.0,0.0,4988.954391
2,2022-09-20 02:00:00,3812.128396,5075.091816,5704.491937,3812.128396,3812.128396,2280.964554,2280.964554,2280.964554,-631.183411,...,3.316975,3.316975,3.316975,2.007545,2.007545,2.007545,0.0,0.0,0.0,5383.795107
3,2022-09-20 03:00:00,3812.128529,4959.874072,5573.248124,3812.128529,3812.128529,2350.623100,2350.623100,2350.623100,-841.413617,...,-8.261207,-8.261207,-8.261207,2.023354,2.023354,2.023354,0.0,0.0,0.0,5249.525891
4,2022-09-20 04:00:00,3812.128662,4816.719844,5449.248700,3812.128662,3812.128662,2236.636388,2236.636388,2236.636388,-867.108420,...,-2.472116,-2.472116,-2.472116,2.039111,2.039111,2.039111,0.0,0.0,0.0,5134.616660
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
163,2022-09-26 19:00:00,3812.149823,6818.332464,7442.463649,3812.149823,3812.149823,4725.346278,4725.346278,4725.346278,-1404.363391,...,-14.050299,-14.050299,-14.050299,3.725470,3.725470,3.725470,0.0,0.0,0.0,7122.233035
164,2022-09-26 20:00:00,3812.149956,7036.907183,7676.179611,3812.149956,3812.149956,5333.275411,5333.275411,5333.275411,-1743.067612,...,-21.125854,-21.125854,-21.125854,3.730439,3.730439,3.730439,0.0,0.0,0.0,7355.890738
165,2022-09-26 21:00:00,3812.150090,6182.726037,6807.844623,3812.150090,3812.150090,4712.681087,4712.681087,4712.681087,-1967.313166,...,-38.493128,-38.493128,-38.493128,3.735339,3.735339,3.735339,0.0,0.0,0.0,6487.574518
166,2022-09-26 22:00:00,3812.150223,6798.631336,7405.579937,3812.150223,3812.150223,5105.301986,5105.301986,5105.301986,-1738.395830,...,-38.493128,-38.493128,-38.493128,3.740169,3.740169,3.740169,0.0,0.0,0.0,7105.452629


In [91]:
y_true = df_wind_test['y'].values
y_pred = forecast['yhat'].values

mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)

### Metrics

An evaluation of its performance by assessing how well the model predicted solar energy output by comparing the actual values (y_true) with the predicted values (y_pred). The three (3) metrics for this evaluation are: Mean Absolute Error, Root Mean Squared Error and R-Squared (R²)

Mean Absolute Error (MAE)

In [92]:
mae = round(mae, 2)
mae

287.59

Root Mean Squared Error (RMSE)

In [93]:
rmse = float(round(rmse, 2))
rmse

400.78

R-Squared (R²)

In [94]:
r2 = round(r2, 4)
r2

0.9711